<a href="https://colab.research.google.com/github/gallimatias7/pre-entrega-1-nlp-gallimatias/blob/main/notebooks/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Configuración de semilla y dispositivo
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Dispositivo en uso: {device}")

# 2. Carga y preparación de datos sintéticos
X_raw, y_raw = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1).to(device)

# 3. Definición de la arquitectura base
model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
    nn.Sigmoid()
).to(device)

# 4. Función de pérdida y optimizador Adam
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 5. Función de evaluación
def evaluate(model, X, y, criterion):
    model.eval()
    with torch.no_grad():
        preds = model(X)
        val_loss = criterion(preds, y).item()
        predicted_classes = (preds >= 0.5).float()
        acc = (predicted_classes == y).float().mean().item()
    return val_loss, acc

# 6. Ciclo de Entrenamiento
epochs = 50
print("Iniciando entrenamiento...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        val_loss, val_acc = evaluate(model, X_val_t, y_val_t, criterion)
        print(f"Época [{epoch+1}/{epochs}] | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Dispositivo en uso: cpu
Iniciando entrenamiento...
Época [10/50] | Train Loss: 0.5049 | Val Loss: 0.5234 | Val Acc: 0.8150
Época [20/50] | Train Loss: 0.3617 | Val Loss: 0.4119 | Val Acc: 0.8250
Época [30/50] | Train Loss: 0.3048 | Val Loss: 0.3840 | Val Acc: 0.8400
Época [40/50] | Train Loss: 0.2936 | Val Loss: 0.3917 | Val Acc: 0.8300
Época [50/50] | Train Loss: 0.2869 | Val Loss: 0.3962 | Val Acc: 0.8250
